In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/trustmesh_economic_data.csv",
    parse_dates=["month"],
    keep_default_na=False
)

print(f"Dataset shape: {df.shape}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

Dataset shape: (12000, 32)
Missing values: 0
Duplicate rows: 0


In [2]:


df["cash_flow_margin"] = (
    df["adjusted_cash_flow"] / df["adjusted_income"] * 100
).round(2)

df["savings_rate"] = (
    df["savings_amount"] / df["adjusted_income"] * 100
).clip(0, 100).round(2)

df["expense_to_income_ratio"] = (
    df["adjusted_expenses"] / df["adjusted_income"] * 100
).round(2)

df["income_to_expense_ratio"] = (
    df["adjusted_income"] / df["adjusted_expenses"]
).round(2)

df[
    [
        "adjusted_income",
        "adjusted_expenses",
        "adjusted_cash_flow",
        "cash_flow_margin",
        "savings_rate",
        "expense_to_income_ratio",
        "income_to_expense_ratio"
    ]
].head()

,adjusted_income,adjusted_expenses,adjusted_cash_flow,cash_flow_margin,savings_rate,expense_to_income_ratio,income_to_expense_ratio
0,26820.52,20097.60,1448.69,5.40,1.49,74.93,1.33
1,29850.89,19325.85,4192.18,14.04,3.22,64.74,1.54
2,31252.98,20911.70,4456.40,14.26,2.73,66.91,1.49
3,38136.96,23891.42,7681.90,20.14,3.12,62.65,1.60
4,36140.97,20059.14,10302.91,28.51,3.89,55.50,1.80


In [3]:


df["payment_behavior_score"] = (
    0.6 * df["payment_discipline_pct"]
    + 0.4 * df["supplier_reliability_pct"]
).round(2)

df["operational_behavior_score"] = (
    0.5 * df["business_continuity_pct"]
    + 0.5 * df["demand_stability_pct"]
).round(2)

df["digital_payment_strength"] = (
    df["digital_payment_pct"] / 100
).round(3)

df[
    [
        "payment_discipline_pct",
        "supplier_reliability_pct",
        "payment_behavior_score",
        "business_continuity_pct",
        "demand_stability_pct",
        "operational_behavior_score",
        "digital_payment_pct",
        "digital_payment_strength"
    ]
].head()

,payment_discipline_pct,supplier_reliability_pct,payment_behavior_score,business_continuity_pct,demand_stability_pct,operational_behavior_score,digital_payment_pct,digital_payment_strength
0,70.84,77.22,73.39,77.39,73.27,75.33,36.62,0.366
1,70.99,89.33,78.33,83.80,76.29,80.04,23.46,0.235
2,75.25,71.17,73.62,76.42,67.41,71.91,43.26,0.433
3,81.06,82.35,81.58,79.07,73.02,76.04,46.95,0.470
4,74.54,78.87,76.27,77.85,76.99,77.42,36.71,0.367


In [4]:


df["shock_exposure"] = (df["shock_type"] != "None").astype(int)

df["financial_stress_score"] = (
    0.6 * df["expense_burden_pct"]
    + 0.4 * (100 - df["cash_flow_margin"].clip(-100, 100))
).round(2)

df = df.sort_values(["business_id", "month"])

df["reputation_momentum"] = (
    df.groupby("business_id")["economic_reputation_index"]
      .diff()
      .round(2)
)

df["cash_flow_momentum"] = (
    df.groupby("business_id")["adjusted_cash_flow"]
      .diff()
      .round(2)
)

df[
    [
        "business_id",
        "month",
        "shock_type",
        "shock_exposure",
        "financial_stress_score",
        "economic_reputation_index",
        "reputation_momentum",
        "cash_flow_momentum"
    ]
].head(10)

,business_id,month,shock_type,shock_exposure,financial_stress_score,economic_reputation_index,reputation_momentum,cash_flow_momentum
0,B001,2024-01-01,None,0,82.80,57.71,NaN,NaN
1,B001,2024-02-01,None,0,73.23,62.98,5.27,2743.49
2,B001,2024-03-01,None,0,74.44,59.83,-3.15,264.22
3,B001,2024-04-01,None,0,69.53,65.75,5.92,3225.50
4,B001,2024-05-01,None,0,61.90,65.44,-0.31,2621.01
5,B001,2024-06-01,None,0,65.98,64.97,-0.47,-2091.66
6,B001,2024-07-01,None,0,72.13,61.35,-3.62,325.31
7,B001,2024-08-01,None,0,77.77,63.51,2.16,-5096.12
8,B001,2024-09-01,None,0,74.61,60.55,-2.96,2644.78
9,B001,2024-10-01,None,0,71.97,61.91,1.36,2492.52


In [5]:

engineered_features = [
    "cash_flow_margin",
    "savings_rate",
    "expense_to_income_ratio",
    "income_to_expense_ratio",
    "payment_behavior_score",
    "operational_behavior_score",
    "digital_payment_strength",
    "shock_exposure",
    "financial_stress_score",
    "reputation_momentum",
    "cash_flow_momentum"
]

feature_summary = (
    df[engineered_features]
    .describe()
    .T
    .round(2)
)

display(feature_summary)

print(f"\nOriginal features: 32")
print(f"Engineered features: {len(engineered_features)}")
print(f"Total features: {df.shape[1]}")

,count,mean,std,min,25%,50%,75%,max
cash_flow_margin,12000.0,14.63,17.14,-68.21,4.78,15.02,25.36,62.24
savings_rate,12000.0,2.95,2.23,0.00,1.33,2.42,4.02,17.57
expense_to_income_ratio,12000.0,61.20,13.39,28.21,52.13,60.64,69.46,115.19
income_to_expense_ratio,12000.0,1.72,0.40,0.87,1.44,1.65,1.92,3.54
payment_behavior_score,12000.0,78.55,6.05,57.02,74.43,78.53,82.69,99.26
operational_behavior_score,12000.0,79.41,6.23,57.80,75.11,79.39,83.74,99.73
digital_payment_strength,12000.0,0.54,0.16,0.09,0.42,0.53,0.64,1.00
shock_exposure,12000.0,0.18,0.39,0.00,0.00,0.00,0.00,1.00
financial_stress_score,12000.0,70.87,14.46,32.03,61.71,70.41,79.33,128.90
reputation_momentum,11500.0,0.16,5.05,-18.37,-3.17,0.17,3.54,22.32



Original features: 32
Engineered features: 11
Total features: 43


In [6]:
# Replace financial stress score with a bounded stress percentage

df["financial_stress_pct"] = (
    0.6 * df["expense_burden_pct"].clip(0, 100)
    + 0.4 * (100 - df["cash_flow_margin"].clip(0, 100))
).round(2)

df.drop(columns="financial_stress_score", inplace=True)

df["financial_stress_pct"].describe().round(2)

count    12000.00
mean        70.05
std         12.96
min         32.03
25%         61.71
50%         70.41
75%         79.15
max        100.00
Name: financial_stress_pct, dtype: float64

## Feature Engineering Summary

Feature engineering transformed the raw economic observations into interpretable behavioral indicators covering financial health, payment behavior, operational stability, digital adoption, economic shocks, financial stress, and month-over-month momentum.

A total of 11 derived features were created to support the downstream Economic Behavior Engine and reputation intelligence components of TrustMesh AI.

Momentum features contain 500 missing values because the first observation for each of the 500 businesses has no previous month for comparison. These values are expected and will be handled during downstream analysis or modeling.